In [1]:
import glob
import json
import os
import uuid

from dotenv import load_dotenv
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import requests as req

import torch

import boto3
from botocore.client import Config
from boto3.s3.transfer import TransferConfig


from koger_detection.obj_det.predictors import Predictor
from koger_detection.obj_det.mydatasets import S3ImageDataset
from koger_detection.obj_det.engine import collate_fn, worker_init_fn

/home/koger/environments/orthocounting/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/koger/environments/orthocounting/lib/python3.10/site-packages/albumentations/__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.8' (you have '2.0.5'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [2]:
load_dotenv()

True

In [3]:
s3_config = Config(
    signature_version='s3',
    s3={
        'payload_signing_enabled': True,
        'addressing_style': 'path',
        'request_checksum_calculation': 'when_required',
        'response_checksum_validation': 'when_required'
    }
)

# Not sure how or if being used right now
transfer_config = TransferConfig(
    use_threads = True,
    multipart_threshold = 16 * 1024 * 1024,  # 16 MB
)

pathfinder = boto3.client(
    's3',
    config=s3_config,
    endpoint_url='https://s3.arcc.uwyo.edu'
)

In [4]:
base_url = "https://pronghorn-count.arcc.uwyo.edu/api/v1"
survey_uuid = "f4c0b5e0-af30-46fa-a276-6169932b6a34"
user_web_api_key = os.environ.get("USER_WEB_API_KEY")
bucket_name = "pronghorn-count"



image_dataset = S3ImageDataset(base_url, survey_uuid, user_web_api_key, pathfinder, bucket_name,
                               rgb=True, scale=None)

In [5]:
# Assumes we have a local .env file that stores things like ROOT

model_name = "06-18-2025-18-43-48"
model_uuid = "f5f4caad-1fd4-46a7-982e-21d396a36d48"

research_project = "pronghorn-survey"

model_path = os.environ.get("MODEL_PATH")
model_folder = os.path.join(model_path, research_project, "runs", model_name)
model_cfg_file = os.path.join(model_folder, "cfg.json")
model_weights_file = os.path.join(model_folder, "final_model.pth")

with open(model_cfg_file) as f:
    cfg = json.load(f)
    model_cfg = cfg['model']

model_cfg['model_weights_pth'] = model_weights_file

In [6]:
rgb = True

dataloader = torch.utils.data.DataLoader(
            image_dataset, batch_size=1, shuffle=False, 
            num_workers=4)
# TODO!!!!!! rgb being true here actually means it is converted to bgr
# I think should actually stay as rgb
predictor = Predictor(model_cfg, invert_color_channel=False)

In [7]:
with req.Session() as s:
    s.post(f"{base_url}/autenticate/", json={"external-id": user_web_api_key})

    for ind, image in enumerate(dataloader):
        if ind % 150 == 0:
            print(ind)

        image_data = image['image'][0]
        if (image_data.shape[0] == 2) and (image_data.shape[1] == 2):
            # This isn't beautiful. When Dataset can't read image returns a 2x2x3 array.
            print("Skipping unreadable image.")
            continue
                                        
        res = predictor(image_data)
        
        boxes = res['boxes'].to('cpu').numpy().astype(np.uint32)
        scores = res['scores'].to('cpu').numpy()
        labels = res['labels'].to('cpu').detach().numpy()

        for box, score, label in zip(boxes, scores, labels):
            pred_data = {
                "image_id": image['image_uuid'][0],
                "model_id": model_uuid,
                "label": int(label),
                "score": float(score),
                "box_tx": int(box[0]),
                "box_ty": int(box[1]),
                "box_bx": int(box[2]),
                "box_by": int(box[3]),
                "returning": True
            }
            resp = s.post(f"{base_url}/create/prediction", json=pred_data, 
                          headers={"creditials": "include"})



0
150
300
450
600
750
900
1050
1200
1350
1500
1650
1800
1950
2100
2250
2400
2550
2700
2850
3000
3150
3300
3450
3600
3750
3900
4050
4200
4350
4500
4650
4800
4950
5100
5250
5400
5550
5700
